In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
cur_path = "/content/drive/MyDrive/GFD_Elliptic/"
os.chdir(cur_path)
!pwd

/content/drive/MyDrive/GFD_Elliptic


In [ ]:
# reset
!pip uninstall -y torch dgl torchdata numpy

# numpy compatible
!pip install numpy==1.26.4

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.17.2 requires torch==2.2.2, which is not installed.
accelerate 1.12.0 requires torch>=2.0.0, which is not installed.
fastai 2.8.7 requires torch<3,>=1.10, which is not installed.
timm 1.0.25 requires torch, which is not installed.
torchtune 0.6.1 requires torchdata==0.11.0, which is not installed.
peft 0.18.1 requires torch>=1.13.0, which is not installed.
sentence-transformers 5.2.3 requires torch>=1.11.0, which is not installed.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is in

  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl (755.5 MB)
ERROR: Operation cancelled by user
^C


In [4]:
# pytorch stable
!pip install torch==2.2.2

# dependency
!pip install torchdata==0.7.1

# dgl build for torch 2.2
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.2/repo.html

  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl (755.5 MB)
  Using cached torchdata-0.7.1-py3-none-any.whl.metadata (13 kB)
Using cached torchdata-0.7.1-py3-none-any.whl (184 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires torchdata==0.11.0, but you have torchdata 0.7.1 which is incompatible.
Looking in links: https://data.dgl.ai/wheels/torch-2.2/repo.html
  Using cached https://data.dgl.ai/wheels/torch-2.2/dgl-2.4.0-cp312-cp312-manylinux1_x86_64.whl (7.8 MB)


# Xử lý dữ liệu

In [11]:
import numpy as np
import pandas as pd
import torch as th
import dgl

def load_elliptic_data(data_dir):
    print("Loading Elliptic data...")
    # 1. Đọc dữ liệu
    df_features = pd.read_csv(f'elliptic_data/elliptic_txs_features.csv', header=None)
    df_edges = pd.read_csv(f'elliptic_data/elliptic_txs_edgelist.csv')
    df_classes = pd.read_csv(f'elliptic_data/elliptic_txs_classes.csv')

    # Column 0 là Node ID, Column 1 là Time step (1 -> 49)
    # Re-map Node ID thành index từ 0 -> N-1 giống logic trong hàm _get_node_idx
    node_ids = df_features[0].values
    id_to_node = {node_id: idx for idx, node_id in enumerate(node_ids)}

    # 2. Xử lý Features
    # Bỏ cột ID (0) và cột Time (1), lấy từ cột 2 trở đi làm features
    features = df_features.iloc[:, 2:].values
    features = th.FloatTensor(features)

    # Lấy time steps để chia train/test
    time_steps = df_features[1].values

    # 3. Xử lý Labels
    # Class '1' = Illicit (Gian lận/Fraud), '2' = Licit (Bình thường), 'unknown' = Không gán nhãn
    # Chuyển thành Binary classification: Fraud = 1, Normal = 0
    df_classes['class'] = df_classes['class'].map({'unknown': -1, '1': 1, '2': 0})
    labels = df_classes['class'].values
    labels = th.LongTensor(labels)

    # 4. Xử lý Edges
    # Lọc bỏ các cạnh chứa node không có trong file features
    df_edges = df_edges[df_edges['txId1'].isin(id_to_node) & df_edges['txId2'].isin(id_to_node)]
    sources = np.array([id_to_node[node] for node in df_edges['txId1']])
    sinks = np.array([id_to_node[node] for node in df_edges['txId2']])

    # 5. Xây dựng đồ thị DGL
    g = dgl.graph((sources, sinks), num_nodes=len(id_to_node))

    # Elliptic là đồ thị có hướng (dòng tiền), nhưng để GNN truyền thông tin tốt hơn,
    # thường ta thêm cạnh ngược (bidirectional)
    g = dgl.to_bidirected(g)
    g.ndata['feat'] = features
    g.ndata['label'] = labels

    # 6. Tạo Train/Test Mask (Theo Paper gốc: train trên time step 1-34, test trên 35-49)
    # Mask bỏ qua các node 'unknown' (label == -1)
    train_mask = (time_steps <= 34) & (labels.numpy() != -1)
    test_mask = (time_steps > 34) & (labels.numpy() != -1)

    train_mask = th.BoolTensor(train_mask)
    test_mask = th.BoolTensor(test_mask)

    return g, features, labels, train_mask, test_mask, id_to_node

# Xây dựng Mô hình

In [12]:
import torch.nn as nn
import torch.nn.functional as F
from dgl.nn.pytorch import SAGEConv

class FraudSAGE(nn.Module):
    def __init__(self, in_size, hidden_size, out_size, n_layers, dropout):
        super(FraudSAGE, self).__init__()
        self.layers = nn.ModuleList()

        # Lớp input
        self.layers.append(SAGEConv(in_size, hidden_size, 'mean'))

        # Lớp ẩn
        for _ in range(n_layers - 2):
            self.layers.append(SAGEConv(hidden_size, hidden_size, 'mean'))

        # Lớp output
        self.layers.append(SAGEConv(hidden_size, out_size, 'mean'))
        self.dropout = nn.Dropout(dropout)

    def forward(self, g, features):
        h = features
        for i, layer in enumerate(self.layers[:-1]):
            h = layer(g, h)
            h = F.relu(h)
            h = self.dropout(h)

        logits = self.layers[-1](g, h)
        return logits

# visualize

In [ ]:
import os
import time
import torch as th
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve

def get_f1_score(y_true, y_pred):
    cf_m = confusion_matrix(y_true, y_pred)
    precision = cf_m[1,1] / (cf_m[1,1] + cf_m[0,1] + 10e-5)
    recall = cf_m[1,1] / (cf_m[1,1] + cf_m[1,0] + 10e-5)
    f1 = 2 * (precision * recall) / (precision + recall + 10e-5)
    return precision, recall, f1

def evaluate(model, g, features, labels, mask, device):
    model.eval()
    with th.no_grad():
        logits = model(g, features.to(device))
        logits = logits[mask]
        labels = labels[mask]

        preds = th.argmax(logits, axis=1).cpu().numpy()
        labels_np = labels.cpu().numpy()

        precision, recall, f1 = get_f1_score(labels_np, preds)
    return f1
def plot_training_history(losses, f1_scores, output_dir):
    """Vẽ biểu đồ Loss và F1 qua các epochs"""
    plt.figure(figsize=(8, 6))
    plt.plot(losses, label='Training Loss', color='red')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training Loss over Epochs')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'loss.jpg'))
    plt.close()

    plt.figure(figsize=(8, 6))
    plt.plot(f1_scores, label='Test F1 Score', color='blue')
    plt.xlabel('Epochs')
    plt.ylabel('F1 Score')
    plt.title('Test F1 Score over Epochs')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'f1.jpg'))
    plt.close()
    print("Đã lưu loss.jpg và f1.jpg")

def plot_roc_pr_curves(model, g, features, labels, test_mask, device, output_dir):
    """Vẽ đường cong ROC và Precision-Recall"""
    model.eval()
    with th.no_grad():
        logits = model(g, features.to(device))
        # Lấy dự đoán trên tập test
        logits = logits[test_mask]
        labels_test = labels[test_mask].cpu().numpy()

        # Lấy xác suất dự đoán cho class 1 (Gian lận / Illicit)
        probs = th.softmax(logits, dim=1)[:, 1].cpu().numpy()

    # 1. Vẽ ROC Curve
    fpr, tpr, _ = roc_curve(labels_test, probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'roc_curve.png'))
    plt.close()

    # 2. Vẽ PR Curve (Precision-Recall)
    precision, recall, _ = precision_recall_curve(labels_test, probs)
    pr_auc = auc(recall, precision)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='green', lw=2, label=f'PR curve (AUC = {pr_auc:.4f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc="lower left")
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'pr_curve.png'))
    plt.close()
    print("Đã lưu roc_curve.png và pr_curve.png")

# Training Pipeline

In [14]:
import os
import time
import torch as th
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

# Copy hàm get_f1_score từ train.py của bạn
def get_f1_score(y_true, y_pred):
    cf_m = confusion_matrix(y_true, y_pred)
    # Handle trường hợp division by zero bằng cách cộng 10e-5 giống code gốc
    precision = cf_m[1,1] / (cf_m[1,1] + cf_m[0,1] + 10e-5)
    recall = cf_m[1,1] / (cf_m[1,1] + cf_m[1,0] + 10e-5)
    f1 = 2 * (precision * recall) / (precision + recall + 10e-5)
    return precision, recall, f1

def evaluate(model, g, features, labels, mask, device):
    model.eval()
    with th.no_grad():
        logits = model(g, features.to(device))
        logits = logits[mask]
        labels = labels[mask]

        preds = th.argmax(logits, axis=1).cpu().numpy()
        labels_np = labels.cpu().numpy()

        precision, recall, f1 = get_f1_score(labels_np, preds)
    return f1

def train_fg(model, optim, loss_fn, features, labels, g, train_mask, test_mask, device, n_epochs):
    duration = []
    best_loss = float('inf')
    best_model = None

    # Mảng lưu trữ để visualize
    history_loss = []
    history_f1 = []

    for epoch in range(n_epochs):
        model.train()
        tic = time.time()

        logits = model(g, features.to(device))
        loss = loss_fn(logits[train_mask], labels[train_mask].to(device))

        optim.zero_grad()
        loss.backward()
        optim.step()

        duration.append(time.time() - tic)

        test_f1 = evaluate(model, g, features, labels, test_mask, device)

        # Lưu vào lịch sử
        history_loss.append(loss.item())
        history_f1.append(test_f1)

        print("Epoch {:05d}, Time(s) {:.4f}, Loss {:.4f}, Test F1 {:.4f} ".format(
                epoch, np.mean(duration), loss.item(), test_f1))

        if loss.item() < best_loss:
            best_loss = loss.item()
            import copy
            best_model = copy.deepcopy(model)

    return best_model, history_loss, history_f1

# --- MAIN ---
if __name__ == '__main__':
    # ... (Cấu hình và load data giữ nguyên như phần 1 & 2) ...
    data_dir = './elliptic_bitcoin_dataset'
    output_dir = './output'
    n_epochs = 100
    lr = 0.001
    weight_decay = 5e-4
    hidden_size = 64
    n_layers = 2
    dropout = 0.5
    device = th.device('cuda:0' if th.cuda.is_available() else 'cpu')

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print("Thiết bị sử dụng:", device)

    # LƯU Ý: Đảm bảo bạn đã định nghĩa hàm load_elliptic_data() và class FraudSAGE() ở trên
    g, features, labels, train_mask, test_mask, id_to_node = load_elliptic_data(data_dir)
    g = g.to(device)
    in_feats = features.shape[1]
    n_classes = 2

    model = FraudSAGE(in_feats, hidden_size, n_classes, n_layers, dropout).to(device)
    optim = th.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = th.nn.CrossEntropyLoss()

    print("Starting Model training")
    # Lấy thêm lịch sử loss và f1
    best_model, history_loss, history_f1 = train_fg(model, optim, loss_fn, features, labels, g, train_mask, test_mask, device, n_epochs)
    print("Finished Model training")

    # --- BƯỚC VISUALIZE ---
    print("\nĐang tạo các biểu đồ Visualization...")
    plot_training_history(history_loss, history_f1, output_dir)
    plot_roc_pr_curves(best_model, g, features, labels, test_mask, device, output_dir)

    # Lưu Model
    th.save(best_model.state_dict(), os.path.join(output_dir, 'elliptic_model.pth'))
    print("Model saved to", output_dir)

Thiết bị sử dụng: cpu
Loading Elliptic data...
Starting Model training
Epoch 00000, Time(s) 1.2269, Loss 3.5940, Test F1 0.1428 
Epoch 00001, Time(s) 1.2988, Loss 3.2234, Test F1 0.1505 
Epoch 00002, Time(s) 1.3666, Loss 2.7718, Test F1 0.1573 
Epoch 00003, Time(s) 1.3653, Loss 2.2617, Test F1 0.1644 
Epoch 00004, Time(s) 1.3109, Loss 2.1332, Test F1 0.1704 
Epoch 00005, Time(s) 1.2691, Loss 1.8989, Test F1 0.1760 
Epoch 00006, Time(s) 1.2428, Loss 1.5435, Test F1 0.1832 
Epoch 00007, Time(s) 1.2209, Loss 1.3293, Test F1 0.1907 
Epoch 00008, Time(s) 1.2042, Loss 1.3488, Test F1 0.1967 
Epoch 00009, Time(s) 1.1904, Loss 1.1560, Test F1 0.2036 
Epoch 00010, Time(s) 1.2166, Loss 0.9460, Test F1 0.2093 
Epoch 00011, Time(s) 1.2399, Loss 0.9331, Test F1 0.2124 
Epoch 00012, Time(s) 1.2388, Loss 0.7927, Test F1 0.2135 
Epoch 00013, Time(s) 1.2273, Loss 0.7946, Test F1 0.2174 
Epoch 00014, Time(s) 1.2174, Loss 0.7255, Test F1 0.2202 
Epoch 00015, Time(s) 1.2087, Loss 0.6969, Test F1 0.2221 
E